<a href="https://colab.research.google.com/github/mahmoadalmasry2020-cyber/A-full-training-loop/blob/main/A_full_training_loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from huggingface_hub import notebook_login

notebook_login()


In [5]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 776.6 kB/s eta 0:00:00


In [6]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_dataset=load_dataset("nyu-mll/glue","mrpc")
checkpoint="bert-base-uncased"
tokenizer=AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(example):
    return tokenizer(example["sentence1"],example["sentence2"],truncation=True)

tokenized_dataset=raw_dataset.map(tokenize_function,batched=True)
data_collator=DataCollatorWithPadding(tokenizer=tokenizer)

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

mrpc/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  649kB            

mrpc/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 75.7kB            

mrpc/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

In [7]:

tokenized_dataset = tokenized_dataset.remove_columns(
    ["sentence1", "sentence2", "idx"]
)

tokenized_dataset = tokenized_dataset.rename_column(
    "label", "labels"
)
tokenized_dataset.set_format("torch")
tokenized_dataset["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [8]:
from torch.utils.data import DataLoader
train_dataloader=DataLoader(tokenized_dataset["train"],shuffle=True,batch_size=8,collate_fn=data_collator)
eval_dataloader=DataLoader(tokenized_dataset["validation"],batch_size=8,collate_fn=data_collator)

In [9]:
from transformers import AutoModelForSequenceClassification

model=AutoModelForSequenceClassification.from_pretrained(checkpoint,num_labels=2)

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
from torch.optim import AdamW

optimizer=AdamW(model.parameters(), lr=5e-5)

In [11]:
from transformers import get_scheduler

num_epoch=3
num_training_steps=num_epoch * len(train_dataloader)

lr_scheduler=(
    get_scheduler(
        "linear",
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=num_training_steps,
    )
)
print(num_training_steps)

1377


In [12]:
import torch
device=torch.device("cuda") if torch.cuda.is_available else torch.device("cpu")
model.to(device)
device

device(type='cuda')

In [13]:
import sys

sys.modules.pop("torchvision", None)

<module 'torchvision' from '/usr/local/lib/python3.13/dist-packages/torchvision/__init__.py'>

In [14]:
from tqdm.auto import tqdm

progress_bar=tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epoch):
    for batch in train_dataloader:
        batch={k :v.to(device) for k , v in batch.items()}
        output=model(**batch)
        loss=output.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

  0%|          | 0/1377 [00:00<?, ?it/s]

In [21]:
import evaluate
metrics=evaluate.load("glue","mrpc")
model.eval()
for batch in eval_dataloader:
        batch={k :v.to(device) for k , v in batch.items()}
        with torch.no_grad():
            output=model(**batch)

        logits=output.logits
        predictions=torch.argmax(logits,dim=-1)
        metrics.add_batch(predictions=predictions,references=batch["labels"])
metrics.compute()


{'accuracy': 0.8504901960784313, 'f1': 0.897133220910624}

In [20]:
model.push_to_hub("bert-base-uncased-mrpc")
tokenizer.push_to_hub("bert-base-uncased-mrpc")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...kpn6whv/model.safetensors:   0%|          | 14.2kB /  438MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/supterr/bert-base-uncased-mrpc/commit/aa72c7b5ef66a75d719c15ed809a3dfba39dc1bf', commit_message='Upload tokenizer', commit_description='', oid='aa72c7b5ef66a75d719c15ed809a3dfba39dc1bf', pr_url=None, repo_url=RepoUrl('https://huggingface.co/supterr/bert-base-uncased-mrpc', endpoint='https://huggingface.co', repo_type='model', repo_id='supterr/bert-base-uncased-mrpc'), pr_revision=None, pr_num=None)

In [27]:
from transformers import pipeline

classifier=pipeline("text-classification",model="supterr/bert-base-uncased-mrpc")


results=classifier({

    "text":"The company released a new AI model.",
    "text_pair":"A new AI model was released by the company"

})

print(results)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'label': 'LABEL_1', 'score': 0.995075523853302}
